In [ ]:
import os
from dotenv import load_dotenv

# Charge les variables du fichier .env dans l'environnement
load_dotenv()

# Récupère la clé
RAPIDAPI_KEY = os.getenv("cle_gaut")

print(f"Ma clé est bien chargée : {RAPIDAPI_KEY[:4]}****") # Sécurité : n'affiche que le début

Ma clé est bien chargée : b61f****


In [ ]:
import requests
import json

RAPIDAPI_HOST = "google-flights2.p.rapidapi.com"
url = f"https://{RAPIDAPI_HOST}/api/v1/searchFlights"

querystring = {
    "departure_id": "LAX",
    "arrival_id": "JFK",
    "outbound_date": "2026-06-15",
    "travel_class": "ECONOMY",
    "adults": "1",
    "currency": "EUR"
}

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": RAPIDAPI_HOST,
    "Content-Type": "application/json"
}

print(f"Recherche de vols ({querystring['departure_id']} -> {querystring['arrival_id']})...")
print("Attente de la réponse (maximum 15 secondes)...")

try:
    response = requests.get(url, headers=headers, params=querystring, timeout=15)
    response.raise_for_status() 

    data = response.json()
    print(" Données récupérées !")
    
    # --- AJOUT : Sauvegarde dans un fichier local ---
    nom_fichier = "resultats_vols.json"
    with open(nom_fichier, 'w', encoding='utf-8') as fichier:
        # indent=4 permet de formater le fichier pour qu'il soit lisible par un humain
        json.dump(data, fichier, ensure_ascii=False, indent=4)
        
    print(f" Le JSON complet a été sauvegardé dans le fichier : {nom_fichier}")
    # ------------------------------------------------

except requests.exceptions.Timeout:
    print("⏳ Erreur : L'API a mis plus de 15 secondes à répondre (Timeout). Le scraper est trop lent ou bloqué.")
except requests.exceptions.HTTPError as errh:
    print(f"❌ Erreur HTTP : {errh} (Vérifie tes paramètres ou ta clé API)")
except requests.exceptions.RequestException as e:
    print(f"❌ Erreur générale de connexion : {e}")

Recherche de vols (LAX -> JFK)...
Attente de la réponse (maximum 15 secondes)...
✅ Données récupérées !
📁 Le JSON complet a été sauvegardé dans le fichier : resultats_vols.json


In [ ]:
import json
import pandas as pd

# 1. EXTRACTION : Lecture du fichier JSON local
nom_fichier_json = 'resultats_vols.json'
nom_fichier_csv = 'vols_propres.csv'

print(f" Lecture du fichier {nom_fichier_json}...")

try:
    with open(nom_fichier_json, 'r', encoding='utf-8') as fichier:
        json_data = json.load(fichier)
except FileNotFoundError:
    print(f" Erreur : Le fichier {nom_fichier_json} est introuvable.")
    exit()

# 2. TRANSFORMATION : Navigation ciblée dans le JSON
vols_propres = []

try:
    # On descend dans l'arborescence : data -> itineraries
    itineraries = json_data.get("data", {}).get("itineraries", {})
    
    # On récupère les deux listes de vols (s'ils n'existent pas, on renvoie une liste vide [])
    top_flights = itineraries.get("topFlights", [])
    other_flights = itineraries.get("otherFlights", [])
    
    # ASTUCE PYTHON : On fusionne les deux listes pour traiter tous les vols en même temps
    tous_les_vols = top_flights + other_flights
    
    if not tous_les_vols:
        print("Aucune donnée de vol trouvée dans 'topFlights' ou 'otherFlights'.")
        exit()

    for vol in tous_les_vols:
        # Les détails de la compagnie et des aéroports sont dans un sous-tableau "flights"
        segments = vol.get("flights", [])
        
        if not segments:
            continue # S'il n'y a pas de détail de vol, on passe au suivant
            
        # On prend le premier segment pour avoir l'aéroport de départ initial et la compagnie
        premier_segment = segments[0]
        # On prend le dernier segment pour avoir l'aéroport d'arrivée final (utile s'il y a des escales)
        dernier_segment = segments[-1]

        # On extrait avec tolérance aux pannes (.get())
        vol_formate = {
            "Compagnie": premier_segment.get("airline", "N/A"),
            "Depart_Code": premier_segment.get("departure_airport", {}).get("airport_code", "N/A"),
            "Arrivee_Code": dernier_segment.get("arrival_airport", {}).get("airport_code", "N/A"),
            "Heure_Depart": vol.get("departure_time", "N/A"),
            "Heure_Arrivee": vol.get("arrival_time", "N/A"),
            "Duree": vol.get("duration", {}).get("text", "N/A"),
            "Escales": vol.get("stops", 0),
            "Emissions_CO2_kg": premier_segment.get("carbon_emissions", {}).get("CO2e", 0) / 1000 if "carbon_emissions" in premier_segment else "N/A", # Petit bonus écologique
            "Prix": vol.get("price", "N/A")
        }
        vols_propres.append(vol_formate)

    print(f"{len(vols_propres)} vols transformés avec succès.")

except Exception as e:
    print(f"Erreur lors du parsing du JSON : {e}")
    exit()

# 3. CHARGEMENT (LOAD) : Export en CSV pour Power BI
if vols_propres:
    df = pd.DataFrame(vols_propres)
    
    # Nettoyage optionnel : s'assurer que le prix est bien considéré comme un nombre
    df['Prix'] = pd.to_numeric(df['Prix'], errors='coerce')
    
    df.to_csv(nom_fichier_csv, index=False, encoding='utf-8')
    
    print(f"Le tableau a été exporté dans : {nom_fichier_csv}\n")
    print("Aperçu des données :")
    print(df.head())

📖 Lecture du fichier resultats_vols.json...
✅ 34 vols transformés avec succès.
📊 Le tableau a été exporté dans : vols_propres.csv

Aperçu des données :
  Compagnie Depart_Code Arrivee_Code         Heure_Depart  \
0   JetBlue         LAX          JFK  15-06-2026 04:20 PM   
1   JetBlue         LAX          JFK  15-06-2026 08:35 PM   
2  American         LAX          JFK  15-06-2026 09:30 PM   
3     Delta         LAX          JFK  15-06-2026 09:10 PM   
4  Frontier         LAX          JFK  15-06-2026 11:20 PM   

         Heure_Arrivee        Duree  Escales Emissions_CO2_kg  Prix  
0  16-06-2026 12:56 AM  5 hr 36 min        0              N/A   163  
1  16-06-2026 05:10 AM  5 hr 35 min        0              N/A   163  
2  16-06-2026 06:00 AM  5 hr 30 min        0              N/A   237  
3  16-06-2026 05:25 AM  5 hr 15 min        0              N/A   250  
4  16-06-2026 11:51 AM  9 hr 31 min        0              N/A   182  
